In [1]:
import pandas as pd 
from evolvepro.src.metrics import metrics_calc_sing, enrichment_factor, apk, avg_rank, best_rank, top_recall, ndcg
from evolvepro.src.utils import load_dataset 
import numpy as np 

In [13]:
#old ndcg
def old_ndcg(df_labels:pd.DataFrame, df_round: pd.DataFrame | dict, k:int=10)-> float:
    i_gain = (df_labels['activity'] - df_labels['activity'].min())/(df_labels['activity'].max()-df_labels['activity'].min())
    #print(f'This is the old i_gain: {i_gain}')
    IDCG = (i_gain.head(k)/np.log2(np.arange(1,k+1)+1)).sum()
    print(f'This is the old IDCG: {IDCG}')
    gain =  (df_round['activity'] - df_round['activity'].min())/(df_round['activity'].max()-df_round['activity'].min())
    #print(f'This is the old gain: {gain}')
    DCG = (gain.head(k)/np.log2(np.arange(1,k+1)+1)).sum()
    print(f'This is the old DCG: {DCG}')
    NDCG = DCG/IDCG
    return print(f'This is the old NDCG: {NDCG}')


def _minmax(x: np.ndarray) -> np.ndarray:
    """This helper function performs min-max scaling on a NumPy array, returning values in the range ([0, 1])"""
    x_min, x_max = x.min(), x.max()

    if x_max == x_min:
        return np.zeros_like(x, dtype=float)
    
    return (x - x_min) / (x_max - x_min)

#new ndcg 
def new_ndcg(df_labels:pd.DataFrame, df_round: pd.DataFrame | dict, cutoff:float, k:int=10)-> float:
    activity = np.asarray(df_labels['activity'])
    i_gain = np.where(activity >= cutoff, _minmax(activity), 0.0)
    print(f'This is the new i_gain: {i_gain}')
    IDCG = (i_gain[:k]/np.log2(np.arange(1,k+1)+1)).sum()
    print(f'This is the new IDCG: {IDCG}')
    activity_2 = np.asarray(df_round['activity'])
    gain = np.where(activity_2 >= cutoff, _minmax(activity), 0.0)
    print(f'This is the new gain: {gain}')
    DCG = (gain[:k]/np.log2(np.arange(1,k+1)+1)).sum()
    print(f'This is the new DCG: {DCG}')
    NDCG = DCG/IDCG
    return print(f'This is the new NDCG: {NDCG}')

#Andrea's ndcg
def calc_ndcg(merged_dict: dict | pd.DataFrame, cutoff: float, top_k: float = 10, top_k_mode: str = "percent") -> float:
    """
    Parameters:
        merged_dict: Dictionary containing 'y_true' and 'y_pred' arrays.
        cutoff: Threshold to define positive examples in `y_true`.
        top_k: Value defining the top portion of the ranking to evaluate.
        top_k_mode: Interpretation of `top_k`:
            'percent' -- top_k is a percentage of the total items.
            'count'   -- top_k is an absolute number of items.
    """
    y_true = np.asarray(merged_dict['activity'])
    y_score = np.asarray(merged_dict['y_pred'])

    k = (int(np.floor(y_true.shape[0] * (top_k / 100)))
            if top_k_mode == "percent" else top_k)

    #gains = _minmax(y_true)
    gains = np.where(y_true >= cutoff, _minmax(y_true), 0.0)
    ranks = np.argsort(np.argsort(-y_score)) + 1

    if k == "all":
        k = len(ranks)

    ranks_k   = ranks[ranks <= k]
    gains_k   = gains[ranks <= k]
    ranks_fil = ranks_k[gains_k != 0]
    gains_fil = gains_k[gains_k != 0]

    if len(ranks_fil) == 0:
        return 0.0

    dcg = np.sum([g / np.log2(r + 1) for r, g in zip(ranks_fil, gains_fil)])

    ideal_ranks     = np.argsort(np.argsort(-gains)) + 1
    ideal_ranks_k   = ideal_ranks[ideal_ranks <= k]
    ideal_gains_k   = gains[ideal_ranks <= k]
    ideal_ranks_fil = ideal_ranks_k[ideal_gains_k != 0]
    ideal_gains_fil = ideal_gains_k[ideal_gains_k != 0]
    idcg = np.sum([g / np.log2(r + 1) for r, g in zip(ideal_ranks_fil, ideal_gains_fil)])

    return dcg / idcg


**Ideal Case**

In [3]:
variant_dummy = ['A50G', 'E22F', 'D34K', 'C94F', 'L100V', 
                 'V35H', 'P90R', 'W78E', 'D42S', 'M1T']
activity_dummy = [1.5, 1.2, 0.9, 0.8, 0.7, 
                  0.6, 0.5, 0.4, 0.3, 0.1]
dummy_dict = {'variant': variant_dummy, 
                'activity': activity_dummy}
dummy_labels = pd.DataFrame(dummy_dict, columns=dummy_dict.keys())
print(dummy_labels)
#dummy_labels.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/GPR68/dummy_labels.csv', index=False)
dummy_ypred = [0.95, 0.90, 0.85, 0.80, 0.75,
            0.70, 0.65, 0.60, 0.55, 0.50]
dummy_round_dict = {'variant': variant_dummy,
                    'y_pred': dummy_ypred}
dummy_df_round = pd.DataFrame(dummy_round_dict, columns=dummy_round_dict.keys())
print(dummy_df_round)
#dummy_df_round.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1.csv', index=False)

  variant  activity
0    A50G       1.5
1    E22F       1.2
2    D34K       0.9
3    C94F       0.8
4   L100V       0.7
5    V35H       0.6
6    P90R       0.5
7    W78E       0.4
8    D42S       0.3
9     M1T       0.1
  variant  y_pred
0    A50G    0.95
1    E22F    0.90
2    D34K    0.85
3    C94F    0.80
4   L100V    0.75
5    V35H    0.70
6    P90R    0.65
7    W78E    0.60
8    D42S    0.55
9     M1T    0.50


In [4]:
#dummy_labels = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels.csv'
#dummy_round = pd.read_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1.csv')
#output = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_results.csv'
#round_dummy = 'Round Dummy'
#metrics_calc_sing(round_dummy, dummy_labels, dummy_round, output, threshold_hit=0.6, fraction=0.5, k=5)
dummy_labels["true_rank"] = dummy_labels.index+1
labels_activity = dummy_labels[['variant', 'activity', 'true_rank']]
df_merged = pd.merge(dummy_df_round, labels_activity, on='variant', how='left')
df_ordinato = df_merged.sort_values(by='y_pred', ascending=False).reset_index(drop=True)
old_ndcg(labels_activity, df_ordinato, k = 5)
new_ndcg(labels_activity, df_ordinato,cutoff=0.7, k = 5)
calc_ndcg(df_merged, cutoff=0.7, top_k=5, top_k_mode='count')

This is the old i_gain: 0    1.000000
1    0.785714
2    0.571429
3    0.500000
4    0.428571
5    0.357143
6    0.285714
7    0.214286
8    0.142857
9    0.000000
Name: activity, dtype: float64
This is the old IDCG: 2.1625771456576457
This is the old gain: 0    1.000000
1    0.785714
2    0.571429
3    0.500000
4    0.428571
5    0.357143
6    0.285714
7    0.214286
8    0.142857
9    0.000000
Name: activity, dtype: float64
This is the old DCG: 2.1625771456576457
This is the old NDCG: 1.0
This is the new i_gain: [1.         0.78571429 0.57142857 0.5        0.42857143 0.
 0.         0.         0.         0.        ]
This is the new IDCG: 2.1625771456576457
This is the new gain: [1.         0.78571429 0.57142857 0.5        0.42857143 0.
 0.         0.         0.         0.        ]
This is the new DCG: 2.1625771456576457
This is the new NDCG: 1.0


1.0

**Worst Case**

In [5]:
variant_dummyw = ['A50G', 'E22F', 'D34K', 'C94F', 'L100V', 
                 'V35H', 'P90R', 'W78E', 'D42S', 'M1T']
activity_dummyw = [1.5, 1.2, 0.9, 0.8, 0.7, 
                  0.6, 0.5, 0.4, 0.3, 0.1]
dummy_dictw = {'variant': variant_dummyw, 
                'activity': activity_dummyw}
dummy_labelsw = pd.DataFrame(dummy_dict, columns=dummy_dict.keys())
print(dummy_labelsw)
dummy_labelsw.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels_worst.csv', index=False)
dummy_ypredw = [0.50, 0.55, 0.60, 0.65, 0.70,
            0.75, 0.80, 0.85, 0.90, 0.95]
dummy_round_dictw = {'variant': variant_dummyw,
                    'y_pred': dummy_ypredw}
dummy_df_roundw = pd.DataFrame(dummy_round_dictw, columns=dummy_round_dictw.keys())
print(dummy_df_roundw)
#dummy_df_roundw.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1_worst.csv', index=False)

  variant  activity
0    A50G       1.5
1    E22F       1.2
2    D34K       0.9
3    C94F       0.8
4   L100V       0.7
5    V35H       0.6
6    P90R       0.5
7    W78E       0.4
8    D42S       0.3
9     M1T       0.1
  variant  y_pred
0    A50G    0.50
1    E22F    0.55
2    D34K    0.60
3    C94F    0.65
4   L100V    0.70
5    V35H    0.75
6    P90R    0.80
7    W78E    0.85
8    D42S    0.90
9     M1T    0.95


In [28]:
dummy_labelsw = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels_worst.csv'
dummy_roundw = pd.read_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1_worst.csv')
outputw = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_results_worst.csv'
round_dummy = 'Round Dummy'
metrics_calc_sing(round_dummy,dummy_labelsw,dummy_roundw,outputw,threshold_hit=0.6,fraction=0.5,k=5)


This is the df_metrics new             Round Dummy
ef             0.333333
ap             0.040000
avg_rank       8.000000
best_rank      6.000000
top_recall     0.166667
ndcg           0.212010
diversity      5.000000


,Round Dummy
ef,0.333333
ap,0.040000
avg_rank,8.000000
best_rank,6.000000
top_recall,0.166667
ndcg,0.212010
diversity,5.000000


In [6]:
dummy_labelsw["true_rank"] = dummy_labelsw.index+1
labels_activityw = dummy_labelsw[['variant', 'activity', 'true_rank']]
df_mergedw = pd.merge(dummy_df_roundw, labels_activityw, on='variant', how='left')
df_ordinatow = df_mergedw.sort_values(by='y_pred', ascending=False).reset_index(drop=True)
print(df_mergedw)
old_ndcg(labels_activityw, df_ordinatow, k = 5)
new_ndcg(labels_activityw, df_ordinatow,cutoff=0.7, k = 5)
calc_ndcg(df_ordinatow, cutoff=0.7, top_k=5, top_k_mode='count')

  variant  y_pred  activity  true_rank
0    A50G    0.50       1.5          1
1    E22F    0.55       1.2          2
2    D34K    0.60       0.9          3
3    C94F    0.65       0.8          4
4   L100V    0.70       0.7          5
5    V35H    0.75       0.6          6
6    P90R    0.80       0.5          7
7    W78E    0.85       0.4          8
8    D42S    0.90       0.3          9
9     M1T    0.95       0.1         10
This is the old i_gain: 0    1.000000
1    0.785714
2    0.571429
3    0.500000
4    0.428571
5    0.357143
6    0.285714
7    0.214286
8    0.142857
9    0.000000
Name: activity, dtype: float64
This is the old IDCG: 2.1625771456576457
This is the old gain: 0    0.000000
1    0.142857
2    0.214286
3    0.285714
4    0.357143
5    0.428571
6    0.500000
7    0.571429
8    0.785714
9    1.000000
Name: activity, dtype: float64
This is the old DCG: 0.45848784111494256
This is the old NDCG: 0.212009935477014
This is the new i_gain: [1.         0.78571429 0.57142857 0.5

0.0

**Near values**

In [8]:
variant_dummy_near = ['A50G', 'E22F', 'D34K', 'C94F', 'L100V', 
                 'V35H', 'P90R', 'W78E', 'D42S', 'M1T']
activity_dummy_near = [1.5, 1.2, 0.9, 0.8, 0.7, 
                  0.6, 0.5, 0.4, 0.3, 0.1]
dummy_dict_near = {'variant': variant_dummy_near, 
                'activity': activity_dummy_near}
dummy_labels_near = pd.DataFrame(dummy_dict_near, columns=dummy_dict_near.keys())
print(dummy_labels_near)
dummy_labels_near.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels_near.csv', index=False)
dummy_ypred_near = [0.90, 0.93, 0.85, 0.69, 0.68,
            0.65, 0.59, 0.55, 0.50, 0.45]
dummy_round_dict_near = {'variant': variant_dummy_near,
                    'y_pred': dummy_ypred_near}
dummy_df_round_near = pd.DataFrame(dummy_round_dict_near, columns=dummy_round_dict_near.keys())
print(dummy_df_round_near)
#dummy_df_round_near.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1_near.csv', index=False)

  variant  activity
0    A50G       1.5
1    E22F       1.2
2    D34K       0.9
3    C94F       0.8
4   L100V       0.7
5    V35H       0.6
6    P90R       0.5
7    W78E       0.4
8    D42S       0.3
9     M1T       0.1
  variant  y_pred
0    A50G    0.90
1    E22F    0.93
2    D34K    0.85
3    C94F    0.69
4   L100V    0.68
5    V35H    0.65
6    P90R    0.59
7    W78E    0.55
8    D42S    0.50
9     M1T    0.45


In [4]:
dummy_labels = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels_near.csv'
dummy_round = pd.read_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1_near.csv')
output = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_results_near.csv'
round_dummy = 'Round Dummy'
metrics_calc_sing(round_dummy, 
                  dummy_labels, 
                  dummy_round, 
                  output, 
                  threshold_hit=0.6, 
                  fraction=0.5, 
                  k=5)

,Round Dummy
ef,1.666667
ap,1.000000
avg_rank,3.000000
best_rank,1.000000
top_recall,0.833333
ndcg,0.963430


In [14]:
dummy_labels_near["true_rank"] = dummy_labels_near.index+1
labels_activity_near = dummy_labels_near[['variant', 'activity', 'true_rank']]
df_merged_near = pd.merge(dummy_df_round_near, labels_activity_near, on='variant', how='left')
df_ordinato_near = df_merged_near.sort_values(by='y_pred', ascending=False).reset_index(drop=True)
print(df_ordinato_near)
old_ndcg(labels_activity_near, df_ordinato_near, k = 5)
new_ndcg(labels_activity_near, df_ordinato_near,cutoff=0.7, k = 5)
calc_ndcg(df_ordinato_near, cutoff=0.7, top_k=5, top_k_mode='count')

  variant  y_pred  activity  true_rank
0    E22F    0.93       1.2          2
1    A50G    0.90       1.5          1
2    D34K    0.85       0.9          3
3    C94F    0.69       0.8          4
4   L100V    0.68       0.7          5
5    V35H    0.65       0.6          6
6    P90R    0.59       0.5          7
7    W78E    0.55       0.4          8
8    D42S    0.50       0.3          9
9     M1T    0.45       0.1         10
This is the old IDCG: 2.1625771456576457
This is the old DCG: 2.083490664280101
This is the old NDCG: 0.9634295213299805
This is the new i_gain: [1.         0.78571429 0.57142857 0.5        0.42857143 0.
 0.         0.         0.         0.        ]
This is the new IDCG: 2.1625771456576457
This is the new gain: [1.         0.78571429 0.57142857 0.5        0.42857143 0.
 0.         0.         0.         0.        ]
This is the new DCG: 2.1625771456576457
This is the new NDCG: 1.0


0.9634295213299805

**1 False Positive**

In [15]:
variant_dummy_fp = ['A50G', 'E22F', 'D34K', 'C94F', 'L100V', 
                 'V35H', 'P90R', 'W78E', 'D42S', 'M1T']
activity_dummy_fp = [1.5, 1.2, 0.9, 0.8, 0.7, 
                  0.6, 0.5, 0.4, 0.3, 0.1]
dummy_dict_fp = {'variant': variant_dummy_fp, 
                'activity': activity_dummy_fp}
dummy_labels_fp = pd.DataFrame(dummy_dict_fp, columns=dummy_dict_fp.keys())
print(dummy_labels_fp)
dummy_labels_fp.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels_fp.csv', index=False)
dummy_ypred_fp = [0.90, 0.93, 0.85, 0.69, 0.59,
            0.65, 0.68, 0.55, 0.50, 0.45]
dummy_round_dict_fp = {'variant': variant_dummy_fp,
                    'y_pred': dummy_ypred_fp}
dummy_df_round_fp = pd.DataFrame(dummy_round_dict_fp, columns=dummy_round_dict_fp.keys())
print(dummy_df_round_fp)
#dummy_df_round_fp.to_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1_fp.csv', index=False)

  variant  activity
0    A50G       1.5
1    E22F       1.2
2    D34K       0.9
3    C94F       0.8
4   L100V       0.7
5    V35H       0.6
6    P90R       0.5
7    W78E       0.4
8    D42S       0.3
9     M1T       0.1
  variant  y_pred
0    A50G    0.90
1    E22F    0.93
2    D34K    0.85
3    C94F    0.69
4   L100V    0.59
5    V35H    0.65
6    P90R    0.68
7    W78E    0.55
8    D42S    0.50
9     M1T    0.45


In [ ]:
dummy_labels = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_labels_fp.csv'
dummy_round = pd.read_csv('/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_round1_fp.csv')
output = '/home/tigem/m.livero/Desktop/EvolvePro/giacomelli/test_metrics/dummy_results_fp.csv'
round_dummy = 'Round Dummy'
metrics_calc_sing(round_dummy, 
                  dummy_labels, 
                  dummy_round, 
                  output, 
                  threshold_hit=0.6, 
                  fraction=0.5, 
                  k=5)
#paragonare lo zero shot 
#pipeline che combina predizioni zero-shot ad active learning, cercare un 
#framework che superi i limiti di entrambni 
#benchmark di proteingene --> problema evo non ha delle metriche ben definite e chiare 
#che ci dicano dove va good e va bad 

,Round Dummy
ef,1.333333
ap,0.800000
avg_rank,3.400000
best_rank,1.000000
top_recall,0.666667
ndcg,0.937875


In [16]:
dummy_labels_fp["true_rank"] = dummy_labels_fp.index+1
labels_activity_fp = dummy_labels_fp[['variant', 'activity', 'true_rank']]
df_merged_fp = pd.merge(dummy_df_round_fp, labels_activity_fp, on='variant', how='left')
df_ordinato_fp = df_merged_fp.sort_values(by='y_pred', ascending=False).reset_index(drop=True)
print(df_ordinato_fp)
old_ndcg(labels_activity_fp, df_ordinato_fp, k = 5)
new_ndcg(labels_activity_fp, df_ordinato_fp,cutoff=0.7, k = 5)
calc_ndcg(df_ordinato_fp, cutoff=0.7, top_k=5, top_k_mode='count')

  variant  y_pred  activity  true_rank
0    E22F    0.93       1.2          2
1    A50G    0.90       1.5          1
2    D34K    0.85       0.9          3
3    C94F    0.69       0.8          4
4    P90R    0.68       0.5          7
5    V35H    0.65       0.6          6
6   L100V    0.59       0.7          5
7    W78E    0.55       0.4          8
8    D42S    0.50       0.3          9
9     M1T    0.45       0.1         10
This is the old IDCG: 2.1625771456576457
This is the old DCG: 2.0282259775323093
This is the old NDCG: 0.9378745084793357
This is the new i_gain: [1.         0.78571429 0.57142857 0.5        0.42857143 0.
 0.         0.         0.         0.        ]
This is the new IDCG: 2.1625771456576457
This is the new gain: [1.         0.78571429 0.57142857 0.5        0.         0.
 0.28571429 0.         0.         0.        ]
This is the new DCG: 1.9967830854142705
This is the new NDCG: 0.9233349614480658


0.8867644827780463